In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from iminuit import Minuit
from iminuit.cost import LeastSquares

In [4]:
def read_csv(filename):
    """Read CSV from Waveforms"""
    dat = np.genfromtxt(filename, delimiter=",", skip_header=13, names=True)
    time = dat["Time_s"]
    voltage = dat["Channel_1_V"]
    return time, voltage


def find_start(time, voltage, show_plot=True):
    """Find timing of ball crossings"""
    threshold = 1.0  # Voltage threshold to detect crossing
    crossings = np.where((voltage[:-1] < threshold) & (voltage[1:] >= threshold))[0]
    t_pass = time[crossings]
    t_pass_sigma = np.std(np.diff(t_pass)) / np.sqrt(len(t_pass) - 1)
    if show_plot:
        plt.plot(time, voltage, label="Waveform")
        plt.axhline(y=threshold, color="r", linestyle="--", label="Threshold")
        plt.scatter(t_pass, voltage[crossings], color="g", label="Crossings")
        plt.xlabel("Time (s)")
        plt.ylabel("Voltage (V)")
        plt.title("Waveform with Detected Crossings")
        plt.legend()
        plt.show()

    return t_pass, t_pass_sigma

File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_1.csv, Crossing Times: [0.98084 1.12886 1.2847  1.41162 1.53046], Sigma: 0.007529616109072241
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_2.csv, Crossing Times: [1.59304 1.74058 1.89608 2.02262 2.14132], Sigma: 0.007490091788489647
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_3.csv, Crossing Times: [0.98846 1.13668 1.29254 1.41952 1.5385 ], Sigma: 0.0075235812616067516
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_4.csv, Crossing Times: [1.74722 1.89476 2.05016 2.1769  2.29572], Sigma: 0.0074387881237469936
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_5.csv, Crossing Times: [1.02826 1.17606 1.33158 1.4582  1.5768 ], Sigma: 0.007524384941641403
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOn

In [ ]:
def acc_func(t, s0, v0, a):
    """Position as a function of time with constant acceleration"""
    return s0 + v0 * t + 0.5 * a * t**2


def acc_fit(t_data, x_data, x_sigma):
    """Fit position data to constant acceleration model"""
    least_squares = LeastSquares(t_data, x_data, x_sigma, acc_func)

    # Create Minuit object with initial parameter guess
    m = Minuit(least_squares, a=1.0, v0=0, s0=0)  # Initial guess for acceleration

    # Perform the fit
    m.migrad()

    # Print results
    print(m.params)
    print(f"Best fit acceleration: {m.values['a']:.4f} ± {m.errors['a']:.4f}")

    # Optional: Check fit quality
    if m.valid:
        print(f"Chi-square / ndf: {m.fval:.2f} / {len(t_data) - 1}")
    return m.values["a"], m.errors["a"]


# Example usage of acc_fit

In [13]:
# Loading x-position data
excel_path = "C:\\Users\\jeppe\\AppStatLocal\\Project_Folder\\Project\\BallOnIncline\\Ball_data\\BallOnIncline_Group1_data.xlsx"
incline_length = pd.read_excel(
    excel_path, sheet_name="hældning_længder", skiprows=0, usecols="B:G"
)
mean_gateposition = []
sigma_gateposition = []
for i in range(5):
    length = np.mean(
        incline_length[f"mål_{i + 1}"].values - incline_length["åbning"].values
    )
    mean_gateposition.append(length)
    sigma = np.std(
        incline_length[f"mål_{i + 1}"].values - incline_length["åbning"].values, ddof=1
    )
    sigma_gateposition.append(sigma)
mean_gateposition = np.array(mean_gateposition) / 100  # Convert to meters
sigma_gateposition = np.array(sigma_gateposition) / 100  # Convert to meters
print("Mean gate positions:", np.array(mean_gateposition))
print("Sigma gate positions:", np.array(sigma_gateposition))


Mean gate positions: [0.15112 0.26478 0.41502 0.5634  0.7213 ]
Sigma gate positions: [0.00052154 0.00126174 0.00085264 0.00041833 0.00103682]


In [ ]:
# looping over time data and fitting to find acceleration
a_array = []
a_sigma_array = []
for i in range(10):
    filename = f"C:\\Users\\jeppe\\AppStatLocal\\Project_Folder\\Project\\BallOnIncline\\Ball_data\\BallIncline_{i + 1}.csv"
    time, voltage = read_csv(filename)
    t_pass, t_pass_sigma = find_start(time, voltage, show_plot=False)
    print(f"File: {filename}, Crossing Times: {t_pass}, Sigma: {t_pass_sigma}")
    a, a_sigma = acc_fit(t_pass, mean_gateposition, sigma_gateposition)
    a_array.append(a)
    a_sigma_array.append(a_sigma)
a_array = np.array(a_array)
print("Accelerations from all trials:", a_array)

File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_1.csv, Crossing Times: [0.98084 1.12886 1.2847  1.41162 1.53046], Sigma: 0.007529616109072241
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ s0   │   0.157   │   0.018   │            │            │         │         │       │
│ 1 │ v0   │  -0.675   │   0.030   │            │            │         │         │       │
│ 2 │ a    │   1.364   │   0.025   │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
Best fit acceleration: 1.3638 ± 0.0250
Chi-square / ndf: 0.72 / 4
File: C:\Users\jeppe\AppStatLocal\Project_Folder\Project\BallOnIncline\Ball_data\BallIncline_2.csv, Crossing T